# IRC Labels — Write-Path Walkthrough

This notebook drives the `UpdateLabels` REST verb proposed in [labels-crud-followup.md](https://github.com/laskoviymishka/slop/tree/main/projects/iceberg-governance/labels/labels-crud-followup.md) end-to-end against a Lakekeeper built from the [labels-crud-verb branch](https://github.com/laskoviymishka/lakekeeper/tree/labels-crud-verb).

The demo uses direct REST calls — the wire shape is visible in every cell. For the SQL DDL surface that translates to these calls, see [Trino PR 4](https://github.com/laskoviymishka/trino/tree/labels-crud-verb).

## What you'll see

1. `/v1/config` advertises the new endpoints
2. Empty initial state via `GET /labels`
3. Atomic write of table + column labels
4. `403 LabelKeyNotWritable` on catalog-managed key writes
5. Atomicity: one bad entry rejects the whole request
6. Lifecycle invariant: `metadata.json` is not touched

## 0 — Imports and configuration

Configuration is read from environment variables set by `docker-compose.yaml`. Inside the notebook container, Lakekeeper is reachable as `http://lakekeeper:8181`; from your host it's `http://localhost:8181`.

In [ ]:
import json
import os
import sys

import requests

sys.path.insert(0, os.path.dirname(os.path.abspath("irc_labels_client.py")))
from irc_labels_client import (
    IrcLabelsClient,
    READ_ONLY_KEYS,
    remove_column,
    remove_table,
    update_column,
    update_table,
)

LAKEKEEPER_URL = os.environ.get("LAKEKEEPER_URL", "http://lakekeeper:8181")
MINIO_URL = os.environ.get("MINIO_URL", "http://minio:9000")
WAREHOUSE = os.environ.get("WAREHOUSE_NAME", "labels-demo")
NAMESPACE = os.environ.get("NAMESPACE", "sales")
TABLE = os.environ.get("TABLE", "orders")

print(f"Lakekeeper:  {LAKEKEEPER_URL}")
print(f"Warehouse:   {WAREHOUSE}")
print(f"Target:      {NAMESPACE}.{TABLE}")
print(f"Read-only keys (catalog-managed): {READ_ONLY_KEYS}")

## 1 — Bootstrap the warehouse

One-time setup: create the default project, then a warehouse backed by MinIO. Idempotent — re-running is safe.

In [ ]:
PROJECT_ID = "00000000-0000-0000-0000-000000000000"


def bootstrap_project():
    r = requests.post(
        f"{LAKEKEEPER_URL}/management/v1/project",
        json={"project-id": PROJECT_ID, "project-name": "default"},
        timeout=10,
    )
    if r.status_code in (200, 201):
        print(f"Created default project (id={PROJECT_ID})")
    elif r.status_code == 409:
        print(f"Default project already exists (id={PROJECT_ID})")
    else:
        raise RuntimeError(f"Project bootstrap failed: {r.status_code} {r.text}")


def ensure_warehouse():
    r = requests.get(f"{LAKEKEEPER_URL}/management/v1/warehouse", timeout=10)
    r.raise_for_status()
    existing = [
        w for w in r.json().get("warehouses", [])
        if (w.get("warehouse-name") or w.get("name")) == WAREHOUSE
    ]
    if existing:
        print(f"Warehouse {WAREHOUSE!r} already exists (id={existing[0]['id']})")
        return existing[0]["id"]

    r = requests.post(
        f"{LAKEKEEPER_URL}/management/v1/warehouse",
        json={
            "warehouse-name": WAREHOUSE,
            "project-id": PROJECT_ID,
            "storage-profile": {
                "type": "s3",
                "bucket": "warehouse",
                "region": "us-east-1",
                "endpoint": MINIO_URL,
                "path-style-access": True,
                "flavor": "minio",
                "sts-enabled": False,
            },
            "storage-credential": {
                "type": "s3",
                "credential-type": "access-key",
                "aws-access-key-id": "admin",
                "aws-secret-access-key": "password",
            },
        },
        timeout=15,
    )
    if r.status_code not in (200, 201):
        raise RuntimeError(f"Warehouse creation failed: {r.status_code} {r.text}")
    wid = r.json()["id"]
    print(f"Created warehouse {WAREHOUSE!r} (id={wid})")
    return wid


bootstrap_project()
warehouse_id = ensure_warehouse()

## 2 — Capability discovery via `/v1/config`

Engines learn what verbs a catalog supports by reading the `endpoints[]` array in `/v1/config`. The two new entries are `catalog-v1-load-labels` and `catalog-v1-update-labels`.

In [ ]:
client = IrcLabelsClient(base_url=LAKEKEEPER_URL, warehouse=WAREHOUSE)

cfg = client.config()
print("Total endpoints advertised:", len(cfg.get("endpoints", [])))
labels_endpoints = [e for e in cfg["endpoints"] if "labels" in e]
for e in labels_endpoints:
    print("  \u2192", e)

assert client.supports("catalog-v1-load-labels"), "missing load-labels endpoint"
assert client.supports("catalog-v1-update-labels"), "missing update-labels endpoint"
print("\n\u2705 catalog advertises both labels endpoints")

## 3 — Create namespace + table

Use PyIceberg to create the namespace and a table with a schema. The schema's `field-id` values are what column-level label writes target — Iceberg field-ids are stable across schema evolution, unlike column names.

In [ ]:
from pyiceberg.catalog import load_catalog
from pyiceberg.schema import Schema
from pyiceberg.types import LongType, NestedField, StringType

catalog = load_catalog(
    WAREHOUSE,
    **{
        "type": "rest",
        "uri": f"{LAKEKEEPER_URL}/catalog",
        "warehouse": WAREHOUSE,
        "s3.endpoint": MINIO_URL,
        "s3.access-key-id": "admin",
        "s3.secret-access-key": "password",
        "s3.region": "us-east-1",
        "s3.path-style-access": "true",
    },
)
catalog.properties["s3.remote-signing-enabled"] = "false"

try:
    catalog.create_namespace(NAMESPACE)
    print(f"Created namespace {NAMESPACE!r}")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"Namespace {NAMESPACE!r} already exists")
    else:
        raise

schema = Schema(
    NestedField(1, "id", LongType(), required=True),
    NestedField(2, "customer", StringType()),
    NestedField(3, "email", StringType()),
)
table_ident = (NAMESPACE, TABLE)
try:
    table = catalog.create_table(table_ident, schema=schema)
    print(f"Created table {NAMESPACE}.{TABLE}")
except Exception as e:
    if "already exists" in str(e).lower():
        table = catalog.load_table(table_ident)
        print(f"Table {NAMESPACE}.{TABLE} already exists")
    else:
        raise

metadata_location_before = table.metadata_location
print("\nSchema field-ids (used by column-label writes):")
for f in table.schema().fields:
    print(f"  field-id={f.field_id}  name={f.name}  type={f.field_type}")
EMAIL_FIELD_ID = next(f.field_id for f in table.schema().fields if f.name == "email")

## 4 — Initial state: `GET /labels`

Brand-new table has empty labels and column-labels.

In [ ]:
initial = client.load_labels(NAMESPACE, TABLE)
print(json.dumps(initial, indent=2))
assert initial == {"labels": {}, "column-labels": []}, \
    "expected empty split shape on a fresh table"
print("\n\u2705 empty split shape returned")

## 5 — Write table labels (`POST /labels`)

Wire shape:

```json
{
  "updates": [
    {"key": "domain",  "value": "customer"},
    {"key": "tier",    "value": "gold"}
  ],
  "removals": []
}
```

The catalog echoes the full post-update label set.

In [ ]:
status, body = client.update_labels(
    NAMESPACE, TABLE,
    updates=[
        update_table("domain", "customer"),
        update_table("tier", "gold"),
    ],
)
print(f"HTTP {status}")
print(json.dumps(body, indent=2))
assert status == 200
assert body["labels"]["domain"] == "customer"
assert body["labels"]["tier"] == "gold"
print("\n\u2705 two table-level labels written, response echoes them")

## 6 — Write column labels

Column-scoped entries carry `field-id`. The catalog returns them in the `column-labels` array.

```json
{
  "updates": [
    {"field-id": 3, "key": "pii-type", "value": "email"}
  ]
}
```

In [ ]:
status, body = client.update_labels(
    NAMESPACE, TABLE,
    updates=[
        update_column(EMAIL_FIELD_ID, "pii-type", "email"),
        update_column(EMAIL_FIELD_ID, "sensitivity", "high"),
    ],
)
print(f"HTTP {status}")
print(json.dumps(body, indent=2))
assert status == 200
col = next(c for c in body["column-labels"] if c["field-id"] == EMAIL_FIELD_ID)
assert col["labels"]["pii-type"] == "email"
assert col["labels"]["sensitivity"] == "high"
print("\n\u2705 column-labels entry populated for field-id", EMAIL_FIELD_ID)

## 7 — `403 LabelKeyNotWritable`

Catalog-managed keys (e.g. `last-accessed-at`) are read-only from the client. Attempting to write one is rejected with the spec error envelope **before any mutation happens**.

```json
{
  "error": {
    "type":    "LabelKeyNotWritable",
    "message": "Label key 'last-accessed-at' is catalog-managed and cannot be written via the API.",
    "code":    403,
    "stack":   ["key=last-accessed-at"]
  }
}
```

In [ ]:
status, body = client.update_labels(
    NAMESPACE, TABLE,
    updates=[update_table("last-accessed-at", "2026-05-12T00:00:00Z")],
)
print(f"HTTP {status}")
print(json.dumps(body, indent=2))
assert status == 403
err = body.get("error", {})
assert err.get("type") == "LabelKeyNotWritable", err
assert "last-accessed-at" in err.get("message", "")
print("\n\u2705 catalog rejected write to catalog-managed key")

## 8 — Atomicity: one bad entry rejects the whole request

Send a request with one valid update + one update to a read-only key. The spec says **all-or-nothing**: the catalog must reject the entire request and apply no mutation.

We verify by:
1. Snapshotting current labels via `GET`.
2. Sending a mixed request and asserting it returns 403.
3. Re-snapshotting and asserting nothing changed.

In [ ]:
before = client.load_labels(NAMESPACE, TABLE)
print("Before:")
print(json.dumps(before, indent=2))

status, body = client.update_labels(
    NAMESPACE, TABLE,
    updates=[
        update_table("new-key", "this-should-NOT-be-applied"),
        update_table("last-edit-by", "impostor"),
    ],
)
print(f"\nHTTP {status}")
print(json.dumps(body, indent=2))
assert status == 403

after = client.load_labels(NAMESPACE, TABLE)
print("\nAfter (must equal before):")
print(json.dumps(after, indent=2))
assert before == after, "atomicity violated: state changed despite 403"
assert "new-key" not in after["labels"], "valid update leaked through"
print("\n\u2705 entire request rejected atomically — no partial mutation")

## 9 — Mixed update + removal in one call

A single request can carry both `updates` and `removals` lists at table or column scope. The catalog applies them atomically.

Demonstrate: remove `tier`, add `cost-center`, remove the column-level `sensitivity` (keep `pii-type`).

In [ ]:
status, body = client.update_labels(
    NAMESPACE, TABLE,
    updates=[
        update_table("cost-center", "sales-analytics"),
    ],
    removals=[
        remove_table("tier"),
        remove_column(EMAIL_FIELD_ID, "sensitivity"),
    ],
)
print(f"HTTP {status}")
print(json.dumps(body, indent=2))
assert status == 200
assert "tier" not in body["labels"]
assert body["labels"].get("cost-center") == "sales-analytics"
col = next(c for c in body["column-labels"] if c["field-id"] == EMAIL_FIELD_ID)
assert "sensitivity" not in col["labels"]
assert col["labels"]["pii-type"] == "email"
print("\n\u2705 update + removal applied atomically across table and column scope")

## 10 — Lifecycle invariant: `metadata.json` is untouched

The hard contract of IRC labels: label writes are catalog-scoped enrichment, NOT table state. They do not modify `metadata.json`, do not create snapshots, do not appear in time-travel queries.

Verify by comparing the table's `metadata_location` before and after all the label writes above.

In [ ]:
table.refresh()
metadata_location_after = table.metadata_location

print(f"Before all label writes: {metadata_location_before}")
print(f"After  all label writes: {metadata_location_after}")

assert metadata_location_before == metadata_location_after, \
    "label writes leaked into metadata.json — lifecycle invariant violated"
print("\n\u2705 metadata_location unchanged \u2014 labels are NOT table state")

## Recap

We exercised the full IRC label CRUD contract against a real catalog:

- **Discovery** — `/v1/config endpoints[]` advertises the two new verbs
- **Split shape** — `labels` (flat) + `column-labels` (keyed by `field-id`), symmetric for read and write
- **Atomic check-before-mutate** — read-only key validation runs first; one bad entry rejects the whole request
- **`403 LabelKeyNotWritable`** — spec error envelope with the offending key
- **Mixed update + removal** — one call can carry both lists across table and column scope
- **Lifecycle invariant** — `metadata.json` is not modified

## Next steps

- **SQL DDL**: PR 4 (`trinodb/trino` labels-crud-verb) adds `ALTER TABLE ... SET LABEL` that translates to this REST verb.
- **Read path**: parent repo `irc-labels` demonstrates labels surfaced via `LoadTableResponse.labels` (no extra round trip on table load).
- **OpenAPI**: a follow-up commit on the Iceberg PR will land the YAML schema entries.